# Pipeline 2 — IMDb Sentiment Classification

**Owner:** Eman Shahin

## What this notebook does
Fine-tunes both `bert-base-uncased` and `distilbert-base-uncased` on the **IMDb** dataset —
a binary sentiment classification task (positive / negative movie reviews).

This corresponds to the IMDb result reported in the DistilBERT paper (Table 2).

Results are saved to `results/` for the final comparison notebook.

## Expected results (from paper Table 2)
| Model | IMDb |
|-------|------|
| BERT-base | 93.46% |
| DistilBERT | 92.82% |

In [12]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [15]:
import sys
import os
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

Device: cuda


In [27]:
src_path = os.path.join(os.getcwd(), 'src')

In [28]:
def load_module(name, path):
    import importlib.util
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

train_mod  = load_module("train", src_path + "/train.py")
eval_mod   = load_module("model_evaluate", src_path + "/evaluate.py")
utils_mod  = load_module("utils", src_path + "/utils.py")

train_model             = train_mod.train_model
evaluate_model          = eval_mod.evaluate_model
print_model_info        = utils_mod.print_model_info
measure_inference_speed = utils_mod.measure_inference_speed
save_results            = utils_mod.save_results
count_parameters        = utils_mod.count_parameters
model_size_mb           = utils_mod.model_size_mb

In [29]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [16]:
MAX_LENGTH = 512
BATCH_SIZE = 16    
EPOCHS     = 3
LR         = 2e-5
SEED       = 42

torch.manual_seed(SEED)

> IMDb reviews are quite long, so I set the maximum input length to 512 tokens. Because longer inputs use more GPU memory, I lowered the batch size to 16. I trained the model for 3 epochs with a learning rate of 2e-5, which is a standard fine-tuning setup for BERT and DistilBERT. I also fixed the random seed to 42 so that if I rerun the experiment, I get consistent results

In [17]:
raw = load_dataset('imdb')
print(f"Train: {len(raw['train'])} | Test: {len(raw['test'])}")
print(f"Sample: {raw['train'][0]['text'][:200]}...")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train: 25000 | Test: 25000
Sample: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...


> Load IMDb 
25,000 train / 25,000 test movie reviews

Labels **: 0 = negative, 1 = positive

In [18]:
def get_dataloaders(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True,
                         max_length=MAX_LENGTH, padding='max_length')

    tokenized = raw.map(tokenize, batched=True)

    #DistilBERT has no token_type_ids
    cols = ['input_ids', 'attention_mask', 'label']
    if 'distilbert' not in model_name:
        cols.append('token_type_ids')

    tokenized.set_format('torch', columns=cols)
    tokenized = tokenized.rename_column('label', 'labels')

    train_loader = DataLoader(tokenized['train'], batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(tokenized['test'],  batch_size=BATCH_SIZE)
    return train_loader, test_loader

> I defined a helper function to tokenize the IMDb dataset and create PyTorch dataloaders

In [30]:
def run_pipeline(model_name):
    """Fine-tune a model on IMDb, evaluate it, and save results."""
    print(f'\n{"="*50}')
    print(f'Model: {model_name} | Task: IMDb')
    print(f'{"="*50}')

    train_loader, test_loader = get_dataloaders(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    print_model_info(model, model_name)

    history = train_model(model, train_loader, test_loader, DEVICE, epochs=EPOCHS, lr=LR)
    eval_results = evaluate_model(model, test_loader, DEVICE)
    speed = measure_inference_speed(model, test_loader, DEVICE)

    total_params, _ = count_parameters(model)
    results = {
        'model':            model_name,
        'task':             'imdb',
        'test_accuracy':     eval_results['accuracy'],
        'total_parameters': total_params,
        'model_size_mb':    model_size_mb(model),
        'inference_speed':  speed,
        'training_history': history,
    }

    filename = f"imdb_{model_name.replace('/', '_').replace('-', '_')}.json"
    save_results(results, filename)
    print(f"Test accuracy: {eval_results['accuracy']}%")
    return results

> I run the full fine-tuning pipeline for any model name, whether BERT or DistilBERT

In [20]:
#IMDb: BERT-base
bert_imdb = run_pipeline('bert-base-uncased')


Model: bert-base-uncased | Task: IMDb


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


bert-base-uncased
  Total parameters:     109,483,778
  Trainable parameters: 109,483,778
  Estimated size:       417.65 MB


Epoch 1/3 [val]: 100%|██████████| 1563/1563 [12:08<00:00,  2.14it/s]


Epoch 1: loss=0.2904, val_acc=93.55%


...

Epoch 2: loss=0.1484, val_acc=93.99%


Epoch 3/3 [val]: 100%|██████████| 1563/1563 [12:07<00:00,  2.15it/s]


Epoch 3: loss=0.0740, val_acc=94.16%


Evaluating: 100%|██████████| 1563/1563 [12:07<00:00,  2.15it/s]


Results saved to ../results/imdb_bert_base_uncased.json
Accuracy: 94.16%


In [23]:
#IMDb: DistilBERT
distilbert_imdb = run_pipeline('distilbert-base-uncased')


Model: distilbert-base-uncased | Task: IMDb


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


distilbert-base-uncased
  Total parameters:     66,955,010
  Trainable parameters: 66,955,010
  Estimated size:       255.41 MB


Epoch 1/3 [val]: 100%|██████████| 1563/1563 [06:05<00:00,  4.27it/s]


Epoch 1: loss=0.2999, val_acc=92.32%


...

Epoch 2: loss=0.1633, val_acc=92.91%


Epoch 3/3 [val]: 100%|██████████| 1563/1563 [06:04<00:00,  4.29it/s]


Epoch 3: loss=0.0928, val_acc=93.00%


Evaluating: 100%|██████████| 1563/1563 [06:04<00:00,  4.28it/s]


Results saved to ../results/imdb_distilbert_base_uncased.json
Accuracy: 93.0%


***NOTE:*** I changed val_acc to test_accuracy after running the code that's why you can see val_acc in the results instead of test_accuracy

> Summary

In [26]:
# ── Summary ────────────────────────────────────────────────────────────────
import pandas as pd

rows = [
    {'Model': 'BERT-base (paper)',    'IMDb Accuracy': 93.46},
    {'Model': 'DistilBERT (paper)',   'IMDb Accuracy': 92.82},
    {'Model': 'BERT-base (ours)',     'IMDb Accuracy': bert_imdb['test_accuracy']},
    {'Model': 'DistilBERT (ours)',    'IMDb Accuracy': distilbert_imdb['test_accuracy']},
]

df = pd.DataFrame(rows)
print(df.to_string(index=False))

retention = round(distilbert_imdb['test_accuracy'] / bert_imdb['test_accuracy'] * 100, 1)
print(f'DistilBERT retains {retention}% of BERT accuracy on IMDb')

             Model  IMDb Accuracy
 BERT-base (paper)          93.46
DistilBERT (paper)          92.82
  BERT-base (ours)          94.16
 DistilBERT (ours)          93.00

DistilBERT retains 98.8% of BERT accuracy on IMDb
